# TimesFM (foundation model, zero-shot): Bonus

TimesFM is Google's pretrained time-series foundation model (here the 2.5 200M PyTorch
checkpoint). It forecasts **zero-shot** — no training on our data — from each series'
past sales. We load the checkpoint, forecast the validation horizon, and score with the
same real-validation WMAE.

First run downloads the checkpoint (~1 GB) and CPU inference over all series takes a
few minutes. Logged to the shared DagsHub.

In [5]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [6]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from timesfm import ForecastConfig
from timesfm.timesfm_2p5.timesfm_2p5_torch import TimesFM_2p5_200M_torch

from src.features.nn_preprocessing import build_long_df, temporal_split, get_real_validation, FREQ
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("TimesFM_Training")
SPLIT_DATE = "2012-01-01"
print("Tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [7]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
train_df, valid_df, horizon = temporal_split(Y_df, SPLIT_DATE)
real_valid = get_real_validation(train_raw, stores, features, SPLIT_DATE)
print("horizon:", horizon, "| series:", train_df["unique_id"].nunique())

horizon: 43 | series: 3302


## Load and compile the pretrained model

In [8]:
model = TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(ForecastConfig(
    max_context=512,
    max_horizon=horizon,
    normalize_inputs=True,
    infer_is_positive=True,
    per_core_batch_size=32,
))
print("TimesFM ready.")

TimesFM ready.


## Zero-shot forecast + WMAE on real validation

In [9]:
# One input array per series, plus each series' last training date.
uids = train_df["unique_id"].drop_duplicates().tolist()
series_list, last_ds = [], {}
for uid, g in train_df.groupby("unique_id", sort=False):
    g = g.sort_values("ds")
    series_list.append(g["y"].values.astype(float))
    last_ds[uid] = g["ds"].max()

with mlflow.start_run(run_name="TimesFM_ZeroShot"):
    mlflow.log_param("checkpoint", "google/timesfm-2.5-200m-pytorch")
    mlflow.log_param("mode", "zero-shot")
    mlflow.log_param("horizon_weeks", int(horizon))

    point_forecast, _ = model.forecast(horizon=horizon, inputs=series_list)
    point_forecast = np.asarray(point_forecast)

    # Build a long forecast frame and score on the real validation rows.
    rows = []
    for i, uid in enumerate(uids):
        dates = pd.date_range(last_ds[uid] + pd.Timedelta(weeks=1), periods=horizon, freq=FREQ)
        for h in range(horizon):
            rows.append((uid, dates[h], float(point_forecast[i][h])))
    fcst = pd.DataFrame(rows, columns=["unique_id", "ds", "timesfm"])

    merged = real_valid.merge(fcst, on=["unique_id", "ds"], how="inner")
    wmae = calculate_wmae(merged["y"], merged["timesfm"].clip(lower=0), merged["IsHoliday"])
    mlflow.log_metric("WMAE", float(wmae))
    print(f"TimesFM zero-shot WMAE (real validation): {wmae:,.2f}")

TimesFM zero-shot WMAE (real validation): 1,635.12
🏃 View run TimesFM_ZeroShot at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/12/runs/12f0427f286346918c4d6dfe5ee31b71
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/12


## Notes

- Zero-shot: TimesFM was never trained on this data; it forecasts from each series' past.
- WMAE on the real validation rows, comparable to the other models.
- Bonus foundation-model experiment; not expected to beat the tuned N-BEATS.